In [1]:
#Python
import re
import random
import importlib
from collections import Counter
import unicodedata
#Terceros
import pandas as pd
#Este proyecto
import set_paths
import config.paths as pths 
from config.patrones_a_eliminar import Pattern, PATRONES
from src.replace_abrev import tokenizar_direccion, reemplazar_abrevs 


In [2]:
dict_abrevs_path = pths.DICCIONARIOS / "dict_abrev.csv"
abrev_df= pd.read_csv(dict_abrevs_path)
abrev_dict= dict(zip(abrev_df['abrev'], abrev_df['significado']))

## TEST REEMPLAZO ABREVIATURAS

In [3]:
#Test
direcciones_path = pths.DIRECCIONES / "universo_direcciones.csv"
direcciones = pd.read_csv(direcciones_path).iloc[:, :2].drop_duplicates(subset=["DIRECCIÓN GARANTÍA"]).dropna()
#direcciones_test = [direcciones.iloc[random.randint(0, direcciones.shape[0]-1)] for _ in range(10)]
direcciones_test = direcciones["DIRECCIÓN GARANTÍA"].sample(n=10, random_state=1105)
for direccion in direcciones_test:
    tokens = tokenizar_direccion(direccion)
    reemplazadas = reemplazar_abrevs(tokens, abrev_dict)
    print(f"{direccion}")
    print(reemplazadas)

PEDRO RODRIGUEZ TRIANA  19 MZ E EDIF L COND 2 DEPTO (5) 302
['PEDRO', 'RODRIGUEZ', 'TRIANA', '19', 'Mz.', 'E', 'EDIF', 'Lt.', 'COND', '2', 'DEPTO', '5', '302']
PORCION 6 DE LA FRACCION DE TERRENO DE LA PARCELA NO 21Z-1P 1 2
['PORCION', '6', 'DE', 'LA', 'FRACCION', 'DE', 'TERRENO', 'DE', 'LA', 'PARCELA', ' No.', '21', 'Z', '-', '1', 'P', '1', '2']
CASA 36 CONDOMINIO 18 CALLE RIO TECOLUTLA
['CASA', '36', 'CONDOMINIO', '18', 'Calle', 'RIO', 'TECOLUTLA']
CALLE AHUACHAPAN 200
['Calle', 'AHUACHAPAN', '200']
LOTE 4 DE LA MANZANA 4 DEL FRACCIONAMIENTO LOS SAUCES COMUNIDAD DE VILLA UNION ,
['Lt.', '4', 'DE', 'LA', 'Mz.', '4', 'DEL', 'FRACCIONAMIENTO', 'LOS', 'SAUCES', 'COMUNIDAD', 'DE', 'VILLA', 'UNION']
AVENIDA MAGNOLIAS, EDIF. 122 B-1 MARCADA CON EL N° B-102 QUE SE EDIFICARÍA SOBRE EL LT 5
['Av.', 'MAGNOLIAS', 'EDIF.', '122', 'B', '-', '1', 'MARCADA', 'CON', 'EL', 'N', 'B', '-', '102', 'QUE', 'SE', 'EDIFICARÍA', 'SOBRE', 'EL', 'Lt.', '5']
CIRCUITO SON JAROCHO #30 INT. B
['CIRCUITO', 'SON', 'J

## TEST N-GRAMAS

In [4]:
direcciones["tokens"] = direcciones["DIRECCIÓN GARANTÍA"].astype(str).apply(lambda x:reemplazar_abrevs(tokenizar_direccion(x), abrev_dict))

In [5]:
counters = {
    1: Counter(),
    2: Counter(),
    3: Counter(),
    4: Counter(),
    5: Counter(),
    8: Counter()
}

for tokens in direcciones["tokens"]:
    for n, counter in counters.items():
        for i in range(len(tokens)-n+1):
            counter[tuple(tokens[i:i+n])] += 1 # -> Construir N-gramas 
#counters[5].most_common() -> Patrones más comunes por longitud (5)

## TEST ENCONTRAR FRASES

In [6]:
def normalizar(token: str) -> str:
    """
    Normaliza un token para facilitar comparaciones.

    - Convierte a mayúsculas.
    - Elimina acentos.
    - Elimina puntos.
    - Elimina espacios sobrantes.
    """

    token = token.upper().strip().replace(".", "")

    token = "".join(
        c for c in unicodedata.normalize("NFD", token)
        if unicodedata.category(c) != "Mn"
    )

    return token

def match(tokens: list[str], pattern: Pattern, i:int) -> bool:

    """Comprueba si una lista de tokens, a partir del i-ésimo token, coincide con un patrón dado."""

    secuencia = pattern.patron

    if i + len(secuencia) > len(tokens):
        return False

    for j in range(len(secuencia)):
        if normalizar(tokens[i+j]) not in secuencia[j]:
            return False

    return True

def buscar_match(tokens: list[str], pattern: Pattern) -> int | None:
    """
    Busca la primera posición en la que una lista de tokens hace match con un patrón dado.
    Puede recibir la lista completa de tokens
    """

    for i in range(len(tokens)):
        if match(tokens, pattern, i):
            return i
    return None

def eliminar_rango(tokens: list[str], start: int, long: int) -> list[str]:

    return tokens[:start] + tokens[start+long:]

def eliminar_patron(tokens: list[str], pattern: Pattern) -> list[str]:
    """
    Elimina la primera ocurrencia de un patrón en una lista de tokens.
    """

    start = buscar_match(tokens, pattern)
    if start is not None:
        return eliminar_rango(tokens, start, len(pattern.patron))
    else:
        return tokens
    
def limpiar_ruido(tokens: list[str], patterns: list[Pattern] = PATRONES) -> list[str]:
    """
    Recorre toda la lista de patrones, elimina todas las primeras apariciones
    """

    for i in range(len(patterns)):
        tokens = eliminar_patron(tokens, patterns[i])
    return tokens


In [52]:
test_adress = random.randint(0, direcciones.shape[0]-1)
print(direcciones["DIRECCIÓN GARANTÍA"][test_adress])
print(limpiar_ruido(tokens = direcciones["tokens"][test_adress]))
print("")

CIRCUITO VIAL, HACIENDA CALPULALPAN, NUMERO 1433-A, CLOTE 2, MANZANA 36
['CIRCUITO', 'VIAL', 'HACIENDA', 'CALPULALPAN', 'No.', '1433', '-', 'A', 'CLOTE', '2', 'Mz.', '36']

